# Site analysis for Vantor Legion and WorldView: what imagery exists at my sites

Updates the Vantor Legion tab of `html/archive_first_look.html`, the first-look page for the SMEs, from two catalogs: Vantor's own (Discovery API, `[Vantor_STAC_API]` in `credentials.ini`), which shows what has been collected, and CSDA STAC (`maxar-legion`), which shows what has been delivered. The two give a pass different IDs, so the dashboard matches them by sensor and time.

`vantor_legion_supp_tracking.ipynb` is the separate tracking of the request and its downloads.

In [1]:
import sys
sys.path.append('/home/pmontesa/code/csda_summaries/lib')

import importlib
import pandas as pd
import multistac, archive_dashboard
for m in (multistac, archive_dashboard):
    importlib.reload(m)

## Parameters

In [2]:
SITES = '/home/pmontesa/code/csda_summaries/sites/csda_sites_aoi.geojson'
SITE_NAME_FIELD = 'Site Name'
SITE_FILTERS = {'Remote Sensing Domain': 'Optical'}
START, END = '2020-01-01', pd.Timestamp.today().strftime('%Y-%m-%d')   # Legion launched 2024; earlier years confirm there is nothing before
PROVIDERS = {'csda': ['maxar-legion'], 'vantor': ['lg01', 'lg02', 'lg03', 'lg04', 'lg05', 'lg06']}

## Search and publish to the dashboard

No cloud limit, since the SMEs set the limits on the page. Split by year for Vantor's 30 s server timeout. Commit and push the page and `html/archive_first_look_data/` to publish.

In [3]:
sites = multistac.load_sites(SITES, site_name_field=SITE_NAME_FIELD, filters=SITE_FILTERS)
dash = multistac.search(sites, START, END, PROVIDERS, split='year')
print(dash)
archive_dashboard.publish('legion', dash.gdf, sites, SITES, SITE_NAME_FIELD, START, END, dash.log)

searches: 100%|██████████| 2842/2842 [02:58<00:00, 15.88it/s]


SearchResult(6120 rows, 6114 unique items, 0 failed tasks)
Vantor Legion: 4843 acquisition-site rows, 6114 catalog items, 58 sites
/panfs/ccds02/home/pmontesa/code/csda_summaries/html/archive_first_look.html


PosixPath('/panfs/ccds02/home/pmontesa/code/csda_summaries/html/archive_first_look.html')

## WorldView-2 and WorldView-3

Updates the Vantor WorldView tab of `html/archive_first_look.html` from Vantor's own catalog (`wv02`, `wv03-vnir`) since 2020, at the same sites. CSDA holds WorldView data in Earthdata CMR, which this does not check yet. Split by year for Vantor's 30 s server timeout; about 15-20 minutes.

In [4]:
WV_START = '2020-01-01'
dash_wv = multistac.search(sites, WV_START, END, {'vantor': ['wv02', 'wv03-vnir']}, split='year')
print(dash_wv)
archive_dashboard.publish('worldview', dash_wv.gdf, sites, SITES, SITE_NAME_FIELD, WV_START, END, dash_wv.log)

searches: 100%|██████████| 812/812 [02:56<00:00,  4.61it/s]


SearchResult(10325 rows, 10167 unique items, 0 failed tasks)
Vantor WorldView: 10325 acquisition-site rows, 10167 catalog items, 58 sites
/panfs/ccds02/home/pmontesa/code/csda_summaries/html/archive_first_look.html


PosixPath('/panfs/ccds02/home/pmontesa/code/csda_summaries/html/archive_first_look.html')